In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download the `.pkg` and install it
   - Windows: download the `.msi` and install it
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local server is running, use:
   ```bash
   curl http://localhost:11434
   ```

4. If you get a connection refused error, restart the Ollama server with:
   ```bash
   !nohup ollama serve > nohup.out 2>&1 &
   ```


In [27]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — in most cases you can still join, but it depends on the course’s enrollment status and deadline.\n\nIf you want, I can help you figure out the best next step. Usually you should check:\n- whether registration is still open\n- whether there’s a waitlist\n- whether the instructor or department allows late enrollment\n\nIf you want, I can also help you draft a quick message asking if you can join.'

In [28]:
def search(query, num_results=5):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [29]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [30]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

In [31]:
call = response.output[0]

In [32]:
import json

args = json.loads(call.arguments)

In [33]:
result = search(**args)

In [34]:
result_json = json.dumps(result, indent=2)

In [35]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [36]:
messages.append(call)

In [37]:
messages.append(function_call_output)

In [38]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course join late enrollment"}', call_id='call_kbUaWxDGo44bYSwf25lGs1x2', name='search', type='function_call', id='fc_026f634bace9c68b006a316460625481a1ba39e0c194ed3f85', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_kbUaWxDGo44bYSwf25lGs1x2',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly w

In [39]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)
response.output

[ResponseOutputMessage(id='msg_026f634bace9c68b006a31647ae53c81a1aea49499f11d2059', content=[ResponseOutputText(annotations=[], text='Yes — you can still join and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 80)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# The loop

In [53]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]



In [54]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [56]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If your goal is to get a certificate, make sure you submit your project while submissions are still open. You can also start learning and working through the materials even if you discovered it late.

If you want, I can also help you with:
- how to start the course,
- whether you can still get a certificate,
- or the course workflow and deadlines.


In [57]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discover late enrollment can I join course FAQ"}', call_id='call_S2FitaJaod4D7xJSdm3L8iwY', name='search', type='function_call', id='fc_0cc5d1f48beb83fb006a31794ed1c0819d8734c67a1cc01060', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course FAQ"}', call_id='call_a85LFfJbDdDcKke1m

In [58]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [59]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install setup FAQ"}
function_call: search {"query":"Olama local run"}
function_call: search {"query":"Ollama localhost model run command"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Download it from: https://ollama.com/download
   - **macOS**: install the `.pkg`
   - **Windows**: install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
 

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Download it from: https://ollama.com/download\n   - **macOS**: install the `.pkg`\n   - **Windows**: install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ol

In [60]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join the course enrollment registration can I join discovered course"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while the course is still accepting submissions.

Also, you don’t necessarily need a registration confirmation to start learning; you can begin using the course materials and submit homework while the submission form is open.

Would you like me to share the best way to get started with the course?


'Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while the course is still accepting submissions.\n\nAlso, you don’t necessarily need a registration confirmation to start learning; you can begin using the course materials and submit homework while the submission form is open.\n\nWould you like me to share the best way to get started with the course?'

In [61]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit meaning chess opening definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a **chess opening** that starts with:

1. **d4 d5**
2. **c4**

It’s called a “gambit” because White offers the c-pawn, hoping to gain better control of the center.

A common idea is:
- White tries to draw Black’s d-pawn away from the center
- then build strong central control with pieces and pawns

There are two main forms:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you meant the **Netflix show** *The Queen’s Gambit*, I can explain that too. Would you like a chess explanation, the show, or both?


'The **Queen’s Gambit** is a **chess opening** that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nIt’s called a “gambit” because White offers the c-pawn, hoping to gain better control of the center.\n\nA common idea is:\n- White tries to draw Black’s d-pawn away from the center\n- then build strong central control with pieces and pawns\n\nThere are two main forms:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you meant the **Netflix show** *The Queen’s Gambit*, I can explain that too. Would you like a chess explanation, the show, or both?'

In [62]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit course FAQ gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit.” If you meant the chess opening, that’s outside the course scope, so I can’t answer it from the FAQ.

If you meant something course-related, please rephrase it and I’ll look it up. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit.” If you meant the chess opening, that’s outside the course scope, so I can’t answer it from the FAQ.\n\nIf you meant something course-related, please rephrase it and I’ll look it up. Are there other areas you want to explore?'

# Frame work

In [69]:
!uv add toyaikit

Resolved 137 packages in 3.30s
Prepared 7 packages in 6.49s
Installed 7 packages in 797ms
 + anthropic==0.109.2
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [70]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
agent_tools = Tools()

In [72]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [73]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [74]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [75]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [76]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [77]:
result.cost

CostInfo(input_cost=Decimal('0.00360225'), output_cost=Decimal('0.0013455'), total_cost=Decimal('0.00494775'))

In [78]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


<class 'dict'>
{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph